In [3]:
"""
engine.py — core engine for the T2I layout game (MVP).

Tier 0  surrogate          : instant, decomposable per-cell feedback
Tier 1  test score         : tiny CNN (torch) or pooled-MLP fallback, val acc + bootstrap CI
Tier 2  certified score    : locked test set (call certified_score offline/cron)

See the invariance note: layout matters ONLY through spatial operators.
"""
import json, math, time
from dataclasses import dataclass
import numpy as np

# ----------------------------- data -----------------------------

@dataclass
class TabularData:
    X_train: np.ndarray; y_train: np.ndarray
    X_val:   np.ndarray; y_val:   np.ndarray
    X_test:  np.ndarray; y_test:  np.ndarray   # locked: tier-2 only
    feature_names: list
    meta: dict = None                          # e.g. ground-truth layout for scrambled_digits
    @property
    def F(self): return self.X_train.shape[1]

def _split(X, y, test=0.15, val=0.15, seed=0):
    from sklearn.model_selection import train_test_split
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=test+val,
                                                stratify=y, random_state=seed)
    X_v, X_te, y_v, y_te = train_test_split(X_tmp, y_tmp, test_size=test/(test+val),
                                            stratify=y_tmp, random_state=seed)
    return X_tr, y_tr, X_v, y_v, X_te, y_te

def _make(X, y, names, seed=0, meta=None):
    X = np.asarray(X, np.float32); y = np.asarray(y)
    X_tr, y_tr, X_v, y_v, X_te, y_te = _split(X, y, seed=seed)
    return TabularData(X_tr, y_tr, X_v, y_v, X_te, y_te, list(names), meta)

# ------------------------- bundled datasets -------------------------

def load_breast_cancer():
    from sklearn.datasets import load_breast_cancer
    d = load_breast_cancer()
    return _make(d.data, d.target, d.feature_names)

def load_wine():
    from sklearn.datasets import load_wine
    d = load_wine()
    return _make(d.data, d.target, d.feature_names)

def make_xor(n=6000, seed=0):
    """Planted-interaction tutorial: label = XOR of two hidden bits.
    Every feature has ~zero individual correlation with y — the surrogate
    is blind by design; only a spatial model (tier-1) can see the pair."""
    rng = np.random.default_rng(seed)
    a = rng.integers(0, 2, n); b = rng.integers(0, 2, n)
    X = rng.normal(0, 1, (n, 12)).astype(np.float32)
    X[:, 0] = a + rng.normal(0, .15, n)
    X[:, 1] = b + rng.normal(0, .15, n)
    y = a ^ b
    names = ["hidden_a", "hidden_b"] + [f"noise_{i}" for i in range(10)]
    return _make(X, y, names, seed=seed)

def make_scrambled_digits(seed=0):
    """Digits pixels as 64 tabular features, column order randomly permuted.
    A ground-truth layout exists (the original 8x8 image geometry) — use
    recovery_metrics() to measure whether search (human or algorithm) rediscovers it."""
    from sklearn.datasets import load_digits
    d = load_digits()
    rng = np.random.default_rng(seed)
    perm = rng.permutation(64)
    X = d.data[:, perm]
    true_fac = -np.ones(64, int)
    for j in range(64):
        true_fac[perm[j]] = j                      # feature j sits at original pixel perm[j]
    names = [f"v{j:02d}" for j in range(64)]       # anonymous on purpose
    return _make(X, d.target, names, seed=seed, meta={"true_layout": true_fac})

def load_csv(path_or_buffer, target_col, seed=0, max_cardinality=50):
    import pandas as pd
    df = pd.read_csv(path_or_buffer).dropna(subset=[target_col])
    y = pd.factorize(df.pop(target_col))[0]
    for c in df.select_dtypes(include=["object", "category"]).columns:
        if df[c].nunique() <= max_cardinality:
            df[c] = df[c].astype("category").cat.codes
        else:
            df = df.drop(columns=c)                # too many levels: drop or target-encode upstream
    df = df.replace([np.inf, -np.inf], np.nan).fillna(df.median(numeric_only=True))
    return _make(df.to_numpy(np.float32), y, list(df.columns), seed=seed)

# --------------------------- layout & rendering ---------------------------

def grid_dims(F, slack=0.2):
    """Smallest square grid with <=20% empty cells (prevents symmetry farming)."""
    w = math.ceil(math.sqrt(F))
    for k in range(w, w + 3):
        if k * k - F <= slack * F:
            return k, k
    return w, w

def sequential_layout(F, H, W):
    fac = -np.ones(H * W, int); fac[:F] = np.arange(F); return fac

def random_layout(F, H, W, rng):
    fac = -np.ones(H * W, int)
    cells = rng.choice(H * W, size=F, replace=False)
    fac[cells] = rng.permutation(F)
    return fac

class SimpleRenderer:
    """Value-in, pixel-out: per-feature robust min-max scaling (train stats ONLY),
    then drop each feature's value into its cell. Swap this class for your T2I
    library's render step when you need paper-pipeline parity."""
    def __init__(self, data: TabularData, H, W, robust=True):
        self.H, self.W = H, W
        X = data.X_train
        lo = np.quantile(X, 0.005, axis=0) if robust else X.min(0)
        hi = np.quantile(X, 0.995, axis=0) if robust else X.max(0)
        rng = np.maximum(hi - lo, 1e-12)
        self.lo, self.scale = lo, 1.0 / rng
    def render(self, X, feature_at_cell):
        Xs = np.clip((X - self.lo) * self.scale, 0, 1).astype(np.float32)
        img = np.zeros((X.shape[0], self.H * self.W), np.float32)
        m = feature_at_cell >= 0
        img[:, m] = Xs[:, feature_at_cell[m]]
        return img.reshape(X.shape[0], self.H, self.W)

def save_layout_dict(feature_names, feature_at_cell, H, W):
    return {"H": H, "W": W,
            "cells": {feature_names[f]: [int(i // W), int(i % W)]
                      for i, f in enumerate(feature_at_cell) if f >= 0}}

def load_layout_json(path_or_stream, feature_names):
    """Integration point #1: import layouts exported from YOUR T2I library
    (or DeepInsight/IGTD) to seed the board / post algorithm-division baselines."""
    d = json.load(path_or_stream)
    H, W = d["H"], d["W"]
    idx = {n: i for i, n in enumerate(feature_names)}
    fac = -np.ones(H * W, int)
    for name, (r, c) in d["cells"].items():
        fac[r * W + c] = idx[name]
    return fac, H, W

def canonical_key(feature_at_cell, H, W):
    """Dihedral-canonical form so mirrored/rotated duplicates don't spam the board."""
    g = feature_at_cell.reshape(H, W)   # square grids only in MVP
    ops = [lambda a: a, np.rot90, lambda a: np.rot90(a, 2), lambda a: np.rot90(a, 3),
           np.fliplr, np.flipud, lambda a: np.fliplr(np.rot90(a)), lambda a: np.flipud(np.rot90(a))]
    return min(tuple(op(g).ravel()) for op in ops)

def recovery_metrics(found_fac, true_fac, H, W):
    """For scrambled_digits: exact-placement % and adjacency-overlap %."""
    exact = float((found_fac == true_fac).mean())
    def pairs(fac):
        g = fac.reshape(H, W); s = set()
        for r in range(H):
            for c in range(W):
                if g[r, c] >= 0:
                    for dr, dc in ((0, 1), (1, 0)):
                        if r + dr < H and c + dc < W and g[r + dr, c + dc] >= 0:
                            s.add(tuple(sorted((g[r, c], g[r + dr, c + dc]))))
        return s
    pf, pt = pairs(found_fac), pairs(true_fac)
    return {"exact_placement": exact, "adjacency_overlap": len(pf & pt) / max(len(pt), 1)}

# --------------------------- tier 0: surrogate ---------------------------

class Surrogate:
    """Instant + decomposable: score = sum(placed importance) + lambda * sum(adjacent-pair synergy).
    Synergy proxy |corr| * min(imp_i, imp_j) is the same prior DeepInsight/REFINED formalize.
    Blind to XOR-style interactions BY DESIGN — that's what spending a Test credit is for."""
    def __init__(self, data: TabularData):
        from sklearn.feature_selection import f_classif
        F, _ = f_classif(data.X_train, data.y_train)
        imp = np.nan_to_num(F, nan=0.0)
        self.imp = imp / (imp.max() + 1e-9)
        C = np.nan_to_num(np.corrcoef(data.X_train, rowvar=False))
        self.syn = np.abs(C) * np.minimum.outer(self.imp, self.imp)
        np.fill_diagonal(self.syn, 0.0)
    def score(self, feature_at_cell, H, W, lam=0.5):
        grid = feature_at_cell.reshape(H, W)
        contrib = np.zeros(H * W); total = 0.0
        for r in range(H):
            for c in range(W):
                f = grid[r, c]
                if f < 0: continue
                s = self.imp[f]
                for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                    r2, c2 = r + dr, c + dc
                    if 0 <= r2 < H and 0 <= c2 < W and grid[r2, c2] >= 0:
                        s += 0.5 * lam * self.syn[f, grid[r2, c2]]   # /2 so cells sum to total
                contrib[r * W + c] = s; total += s
        return total, contrib.reshape(H, W)

# --------------------------- tier 1: test score ---------------------------

def bootstrap_ci(y_true, y_pred, B=400, seed=0):
    rng = np.random.default_rng(seed)
    correct = (y_pred == y_true); n = len(y_true); accs = np.empty(B)
    for b in range(B):
        accs[b] = correct[rng.integers(0, n, n)].mean()
    return tuple(np.percentile(accs, [2.5, 97.5]))

def cnn_score(data, feature_at_cell, H, W, epochs=12, max_train=8000, seed=0, split="val"):
    import torch, torch.nn as nn
    torch.manual_seed(seed)
    ren = SimpleRenderer(data, H, W)
    n = min(max_train, len(data.X_train))
    Xtr = ren.render(data.X_train[:n], feature_at_cell)[:, None]
    ytr = data.y_train[:n]
    Xev = ren.render(data.X_val if split == "val" else data.X_test, feature_at_cell)[:, None]
    yev = np.asarray(data.y_val if split == "val" else data.X_test and data.y_test)
    yev = np.asarray(data.y_val if split == "val" else data.y_test)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    k = len(np.unique(ytr))
    model = nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(2), nn.Flatten(), nn.Linear(32 * 4, k)).to(dev)
    xt, yt = torch.tensor(Xtr).to(dev), torch.tensor(ytr).long().to(dev)
    opt = torch.optim.Adam(model.parameters(), 1e-3)
    lossf = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        perm = torch.randperm(len(xt), device=dev)
        for i in range(0, len(xt), 128):
            b = perm[i:i + 128]
            opt.zero_grad(); loss = lossf(model(xt[b]), yt[b]); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(Xev).to(dev)).argmax(1).cpu().numpy()
    return _report(yev, pred)

def pooled_mlp_score(data, feature_at_cell, H, W, seed=0, max_train=8000):
    """No-torch fallback. 2x2 mean-pooling mixes neighbors -> layout-sensitive
    (plain LR on pixels would score every layout identically — see header note)."""
    from sklearn.neural_network import MLPClassifier
    ren = SimpleRenderer(data, H, W)
    def pool(img):
        n, h, w = img.shape
        h2, w2 = h + h % 2, w + w % 2
        p = np.zeros((n, h2, w2), np.float32); p[:, :h, :w] = img
        return p.reshape(n, h2 // 2, 2, w2 // 2, 2).mean(axis=(2, 4)).reshape(n, -1)
    n = min(max_train, len(data.X_train))
    clf = MLPClassifier((64, 32), max_iter=150, random_state=seed)\
        .fit(pool(ren.render(data.X_train[:n], feature_at_cell)), data.y_train[:n])
    yev = np.asarray(data.y_val)
    pred = clf.predict(pool(ren.render(data.X_val, feature_at_cell)))
    return _report(yev, pred)

def _report(yev, pred):
    from sklearn.metrics import balanced_accuracy_score
    acc = float((pred == yev).mean())
    return {"val_acc": acc, "ci": bootstrap_ci(yev, pred),
            "balanced_acc": float(balanced_accuracy_score(yev, pred)), "n_val": int(len(yev))}

def test_score(data, feature_at_cell, H, W, **kw):
    try:
        import torch  # noqa: F401
        return cnn_score(data, feature_at_cell, H, W, **kw)
    except ImportError:
        return pooled_mlp_score(data, feature_at_cell, H, W, **kw)

def certified_score(data, feature_at_cell, H, W):
    """Tier 2: locked test set. Run offline/season-end under the same protocol
    you use for DeepInsight/IGTD/your method, then compare."""
    try:
        import torch  # noqa: F401
        return cnn_score(data, feature_at_cell, H, W, split="test")
    except ImportError:
        return pooled_mlp_score(data, feature_at_cell, H, W)

# --------------------------- leaderboard ---------------------------

class Leaderboard:
    def __init__(self, path="leaderboard.json"):
        self.path = path
        try: self.rows = json.load(open(path))
        except Exception: self.rows = []
    def submit(self, **row):
        row["ts"] = time.strftime("%Y-%m-%d %H:%M")
        self.rows.append(row)
        json.dump(self.rows, open(self.path, "w"), indent=2)
    def top(self, dataset=None, n=10):
        rows = [r for r in self.rows if dataset is None or r["dataset"] == dataset]
        return sorted(rows, key=lambda r: -r["val_acc"])[:n]

# --------------------------- self-test ---------------------------

if __name__ == "__main__":
    print("T2I layout game — self-test (checks that val acc depends on layout at all)")
    for name, loader in [("breast_cancer", load_breast_cancer), ("xor_tutorial", make_xor)]:
        data = loader(); H, W = grid_dims(data.F)
        rng = np.random.default_rng(0)
        layouts = {"sequential": sequential_layout(data.F, H, W),
                   "random-1": random_layout(data.F, H, W, rng),
                   "random-2": random_layout(data.F, H, W, rng)}
        print(f"\n[{name}]  F={data.F}  grid={H}x{W}")
        for lname, fac in layouts.items():
            s = Surrogate(data).score(fac, H, W)[0]
            r = test_score(data, fac, H, W)
            print(f"  {lname:11s} surrogate={s:7.2f}  val_acc={r['val_acc']:.3f}  "
                  f"CI {r['ci'][0]:.3f}-{r['ci'][1]:.3f}")
    print("\nIf val_acc barely moves across layouts, this dataset has no layout signal — pick another.")

T2I layout game — self-test (checks that val acc depends on layout at all)

[breast_cancer]  F=30  grid=6x6
  sequential  surrogate=  11.67  val_acc=0.894  CI 0.824-0.965
  random-1    surrogate=  11.05  val_acc=0.906  CI 0.847-0.965
  random-2    surrogate=  10.93  val_acc=0.918  CI 0.859-0.976

[xor_tutorial]  F=12  grid=4x4
  sequential  surrogate=   3.40  val_acc=0.992  CI 0.986-0.998
  random-1    surrogate=   3.39  val_acc=0.994  CI 0.990-0.999
  random-2    surrogate=   3.38  val_acc=0.997  CI 0.992-1.000

If val_acc barely moves across layouts, this dataset has no layout signal — pick another.


In [ ]:
"""engine.py — desktop research tools for FeatureFold. NOT shipped in the APK
(excluded via buildozer.spec). sklearn/torch are imported lazily inside functions."""
import json, math
from dataclasses import dataclass
import numpy as np

@dataclass
class TabularData:
    X_train: np.ndarray; y_train: np.ndarray
    X_val: np.ndarray;   y_val: np.ndarray
    X_test: np.ndarray;  y_test: np.ndarray
    feature_names: list
    meta: dict = None
    @property
    def F(self): return self.X_train.shape[1]

def _split(X, y, test=0.15, val=0.15, seed=0):
    from sklearn.model_selection import train_test_split
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=test + val,
                                                stratify=y, random_state=seed)
    X_v, X_te, y_v, y_te = train_test_split(X_tmp, y_tmp, test_size=test / (test + val),
                                            stratify=y_tmp, random_state=seed)
    return X_tr, y_tr, X_v, y_v, X_te, y_te

def _make(X, y, names, seed=0, meta=None):
    X = np.asarray(X, np.float32); y = np.asarray(y)
    X_tr, y_tr, X_v, y_v, X_te, y_te = _split(X, y, seed=seed)
    return TabularData(X_tr, y_tr, X_v, y_v, X_te, y_te, list(names), meta)

def load_breast_cancer():
    from sklearn.datasets import load_breast_cancer
    d = load_breast_cancer()
    return _make(d.data, d.target, d.feature_names)

def load_wine():
    from sklearn.datasets import load_wine
    d = load_wine()
    return _make(d.data, d.target, d.feature_names)

def make_xor(seed=0, n=6000):
    rng = np.random.default_rng(seed)
    a = rng.integers(0, 2, n); b = rng.integers(0, 2, n)
    X = rng.normal(0, 1, (n, 12)).astype(np.float32)
    X[:, 0] = a + rng.normal(0, .15, n); X[:, 1] = b + rng.normal(0, .15, n)
    return _make(X, a ^ b, ["hidden_a", "hidden_b"] + [f"noise_{i}" for i in range(10)], seed)

def make_scrambled_digits(seed=0):
    from sklearn.datasets import load_digits
    d = load_digits()
    rng = np.random.default_rng(seed)
    perm = rng.permutation(64)
    true_fac = -np.ones(64, int)
    for j in range(64): true_fac[perm[j]] = j
    return _make(d.data[:, perm], d.target, [f"v{j:02d}" for j in range(64)],
                 seed, {"true_layout": true_fac})

def load_csv(path_or_buffer, target_col, seed=0, max_cardinality=50):
    import pandas as pd
    df = pd.read_csv(path_or_buffer).dropna(subset=[target_col])
    y = pd.factorize(df.pop(target_col))[0]
    for c in df.select_dtypes(include=["object", "category"]).columns:
        if df[c].nunique() <= max_cardinality:
            df[c] = df[c].astype("category").cat.codes
        else:
            df = df.drop(columns=c)
    df = df.replace([np.inf, -np.inf], np.nan).fillna(df.median(numeric_only=True))
    return _make(df.to_numpy(np.float32), y, list(df.columns), seed)

# ------------------------------------------------------------- layouts / rendering
def grid_dims(F, slack=0.2):
    w = math.ceil(math.sqrt(F))
    for k in range(w, w + 3):
        if k * k - F <= slack * F: return k, k
    return w, w

def sequential_layout(F, H, W):
    fac = -np.ones(H * W, int); fac[:F] = np.arange(F); return fac

def random_layout(F, H, W, rng):
    fac = -np.ones(H * W, int)
    cells = rng.choice(H * W, size=F, replace=False)
    fac[cells] = rng.permutation(F)
    return fac

class SimpleRenderer:
    """Value-in, pixel-out. Swap for YOUR T2I library's render step when you need
    paper-pipeline parity (smoothing/interpolation/channels must match exactly)."""
    def __init__(self, data, H, W, robust=True):
        self.H, self.W = H, W
        X = data.X_train
        lo = np.quantile(X, 0.005, axis=0) if robust else X.min(0)
        hi = np.quantile(X, 0.995, axis=0) if robust else X.max(0)
        self.lo, self.scale = lo, 1.0 / np.maximum(hi - lo, 1e-12)
    def render(self, X, feature_at_cell):
        Xs = np.clip((X - self.lo) * self.scale, 0, 1).astype(np.float32)
        img = np.zeros((X.shape[0], self.H * self.W), np.float32)
        m = feature_at_cell >= 0
        img[:, m] = Xs[:, feature_at_cell[m]]
        return img.reshape(X.shape[0], self.H, self.W)

def save_layout_dict(feature_names, feature_at_cell, H, W):
    return {"H": H, "W": W,
            "cells": {feature_names[f]: [int(i // W), int(i % W)]
                      for i, f in enumerate(feature_at_cell) if f >= 0}}

def load_layout_json(path_or_stream, feature_names):
    """INTEGRATION POINT: import layouts exported by YOUR T2I library /
    DeepInsight / IGTD (algorithm division) or by the mobile app (human division)."""
    d = json.load(path_or_stream)
    H, W = d["H"], d["W"]
    idx = {n: i for i, n in enumerate(feature_names)}
    fac = -np.ones(H * W, int)
    for name, (r, c) in d["cells"].items():
        fac[r * W + c] = idx[name]
    return fac, H, W

def canonical_key(feature_at_cell, H, W):
    g = feature_at_cell.reshape(H, W)
    ops = [lambda a: a, np.rot90, lambda a: np.rot90(a, 2), lambda a: np.rot90(a, 3),
           np.fliplr, np.flipud, lambda a: np.fliplr(np.rot90(a)),
           lambda a: np.flipud(np.rot90(a))]
    return min(tuple(op(g).ravel()) for op in ops)

def recovery_metrics(found_fac, true_fac, H, W):
    exact = float((found_fac == true_fac).mean())
    def pairs(fac):
        g = fac.reshape(H, W); s = set()
        for r in range(H):
            for c in range(W):
                if g[r, c] >= 0:
                    for dr, dc in ((0, 1), (1, 0)):
                        if r + dr < H and c + dc < W and g[r + dr, c + dc] >= 0:
                            s.add(tuple(sorted((g[r, c], g[r + dr, c + dc]))))
        return s
    pf, pt = pairs(found_fac), pairs(true_fac)
    return {"exact_placement": exact,
            "adjacency_overlap": len(pf & pt) / max(len(pt), 1)}

# ------------------------------------------------------------- referee (tier 1)
def stratified_folds(y, K, seed=0):
    rng = np.random.default_rng(seed)
    folds = np.empty(len(y), int)
    for c in np.unique(y):
        idx = np.flatnonzero(y == c); rng.shuffle(idx)
        folds[idx] = np.arange(len(idx)) % K
    return folds

def _fit_linear(D, Y, l2=1e-3, iters=400, lr=0.05):
    n, d = D.shape; k = Y.shape[1]
    Db = np.hstack([D, np.ones((n, 1), np.float32)])
    W = np.zeros((d + 1, k)); m = np.zeros_like(W); v = np.zeros_like(W)
    for t in range(1, iters + 1):
        S = Db @ W; S -= S.max(1, keepdims=True)
        P = np.exp(S); P /= P.sum(1, keepdims=True)
        G = Db.T @ (P - Y) / n; G[:-1] += l2 * W[:-1]
        m = 0.9 * m + 0.1 * G; v = 0.999 * v + 0.01 * G * G
        W -= lr * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)
    return W

def _acc(D, y, W, classes):
    pred = (np.hstack([D, np.ones((len(D), 1), np.float32)]) @ W).argmax(1)
    return float((classes[pred] == y).mean())

def _design(Z, fac, H, W, r=1):
    n = Z.shape[0]
    pix = np.zeros((n, H * W), np.float32)
    m = fac >= 0
    pix[:, m] = Z[:, fac[m]]
    cols = []
    for i in range(H * W):
        r1, c1 = divmod(i, W)
        for j in range(i + 1, H * W):
            r2, c2 = divmod(j, W)
            if max(abs(r1 - r2), abs(c1 - c2)) <= r and fac[i] >= 0 and fac[j] >= 0:
                cols.append((pix[:, i] * pix[:, j])[:, None])
    return np.hstack([pix] + cols) if cols else pix

def _fold_accs(X, y, folds, fac, H, W, r, classes):
    accs = []
    for k in np.unique(folds):
        tr, va = folds != k, folds == k
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
        Dtr = _design((X[tr] - mu) / sd, fac, H, W, r)
        Dva = _design((X[va] - mu) / sd, fac, H, W, r)
        Ytr = np.stack([(y[tr] == c).astype(np.float32) for c in classes], 1)
        accs.append(_acc(Dva, y[va], _fit_linear(Dtr, Ytr), classes))
    return np.array(accs)

def referee_arrays(X, y, fac, H, W, r=1, K=5, seed=0):
    folds = stratified_folds(y, K, seed); classes = np.unique(y)
    a = _fold_accs(X.astype(np.float32), y, folds, fac, H, W, r, classes)
    hw = 2.776 * a.std(ddof=1) / np.sqrt(K)
    return {"acc": float(a.mean()), "folds": a.tolist(),
            "half_width": float(hw), "n_eval": len(y)}

def referee_paired_arrays(X, y, fac_a, fac_b, H, W, r=1, K=5, seed=0):
    """Same folds for both layouts: the paired DIFFERENCE is the well-powered
    quantity (fold difficulty cancels)."""
    folds = stratified_folds(y, K, seed); classes = np.unique(y)
    X = X.astype(np.float32)
    a = _fold_accs(X, y, folds, fac_a, H, W, r, classes)
    b = _fold_accs(X, y, folds, fac_b, H, W, r, classes)
    d = a - b; hw = 2.776 * d.std(ddof=1) / np.sqrt(K)
    return {"mean_diff": float(d.mean()), "per_fold": d.tolist(),
            "ci": (float(d.mean() - hw), float(d.mean() + hw))}

def referee_score(data, fac, H, W, **kw):
    X = np.vstack([data.X_train, data.X_val]); y = np.concatenate([data.y_train, data.y_val])
    return referee_arrays(X, y, fac, H, W, **kw)

def referee_paired(data, fac_a, fac_b, H, W, **kw):
    X = np.vstack([data.X_train, data.X_val]); y = np.concatenate([data.y_train, data.y_val])
    return referee_paired_arrays(X, y, fac_a, fac_b, H, W, **kw)

# ------------------------------------------------------------- gates
def scorer_gate(H=4, W=4, r=1, K=5, min_gap=0.10):
    """GATE 1 — the referee must separate adjacent from scattered placement of a
    planted XOR pair, AND the r=0 control must be flat (the invariance assertion)."""
    rng = np.random.default_rng(0); n = 4000
    a = rng.integers(0, 2, n); b = rng.integers(0, 2, n)
    X = rng.normal(0, 1, (n, H * W)).astype(np.float32)
    X[:, 0] = a + rng.normal(0, .15, n); X[:, 1] = b + rng.normal(0, .15, n)
    y = a ^ b
    def fill(forced):
        fac = -np.ones(H * W, int)
        for cell, f in forced.items(): fac[cell] = f
        rest = iter(f for f in range(H * W) if f not in set(forced.values()))
        for i in range(H * W):
            if fac[i] < 0: fac[i] = next(rest)
        return fac
    adj, anti = fill({0: 0, 1: 1}), fill({0: 0, H * W - 1: 1})
    real = referee_paired_arrays(X, y, adj, anti, H, W, r=r, K=K)
    ctrl = referee_paired_arrays(X, y, adj, anti, H, W, r=0, K=K)
    ok = (real["mean_diff"] >= min_gap and real["ci"][0] > 0
          and abs(ctrl["mean_diff"]) < 0.02)
    return {"ok": ok, "referee": real, "pixels_only_control": ctrl}

def interaction_budget(X, y, K=5, max_pairs=200, seed=0):
    """GATE 2 — upper bound on layout signal: CV gap between an additive model
    and one handed the top pairwise products regardless of layout."""
    X = X.astype(np.float32); y = np.asarray(y)
    folds = stratified_folds(y, K, seed); classes = np.unique(y)
    F = X.shape[1]; scored = []
    for i in range(F):
        for j in range(i + 1, F):
            p = X[:, i] * X[:, j]
            if p.std() < 1e-12: continue
            c = np.corrcoef(p, y)[0, 1]
            scored.append((0.0 if np.isnan(c) else abs(c), i, j))
    scored.sort(reverse=True); keep = scored[:max_pairs]
    def cv(with_products):
        accs = []
        for k in np.unique(folds):
            tr, va = folds != k, folds == k
            mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
            Ztr, Zva = (X[tr] - mu) / sd, (X[va] - mu) / sd
            ctr, cva = [Ztr], [Zva]
            if with_products:
                for _, i, j in keep:
                    ctr.append((Ztr[:, i] * Ztr[:, j])[:, None])
                    cva.append((Zva[:, i] * Zva[:, j])[:, None])
            Ytr = np.stack([(y[tr] == c).astype(np.float32) for c in classes], 1)
            accs.append(_acc(np.hstack(cva), y[va],
                             _fit_linear(np.hstack(ctr), Ytr), classes))
        return float(np.mean(accs))
    a0, a1 = cv(False), cv(True); gap = a1 - a0
    verdict = ("game-worthy" if gap >= 0.03 else
               "weak — sandbox only" if gap >= 0.01 else
               "no layout signal — keep out of the rotation")
    return {"additive": a0, "top_products": a1, "gap": gap, "verdict": verdict}

# ------------------------------------------------------------- certified CNN (tier 2)
def certified_cnn(data, fac, H, W, split="val", runs=3, epochs=8, C=8, seed=0):
    """ONE conv layer: RF = k x k, so a pair interacts nonlinearly only inside a
    window — adjacency binds the hypothesis class. split='test' OFFLINE ONLY."""
    import torch, torch.nn as nn
    kk = 2 if max(H, W) <= 5 else 3
    ren = SimpleRenderer(data, H, W)
    Ye = data.y_val if split == "val" else data.y_test
    Xe = ren.render(data.X_val if split == "val" else data.X_test, fac)[:, None]
    Xt = ren.render(data.X_train, fac)[:, None]
    classes = np.unique(data.y_train); accs = []
    for s in range(seed, seed + runs):
        torch.manual_seed(s)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        net = nn.Sequential(nn.Conv2d(1, C, kk, padding=0), nn.ReLU(),
                            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                            nn.Linear(C, len(classes))).to(dev)
        xt = torch.tensor(Xt).to(dev); yt = torch.tensor(data.y_train).long().to(dev)
        xv = torch.tensor(Xe).to(dev)
        opt = torch.optim.Adam(net.parameters(), 3e-3)
        lf = nn.CrossEntropyLoss(); g = torch.Generator().manual_seed(s)
        for _ in range(epochs):
            perm = torch.randperm(len(xt), generator=g).to(dev)
            for i in range(0, len(xt), 128):
                idx = perm[i:i + 128]
                opt.zero_grad(); lf(net(xt[idx]), yt[idx]).backward(); opt.step()
        with torch.no_grad():
            accs.append(float((classes[net(xv).argmax(1).cpu().numpy()]
                               == np.asarray(Ye)).mean()))
    return {"acc": float(np.mean(accs)), "runs": accs}

if __name__ == "__main__":
    print("=== GATE 1 — scorer sensitivity ===")
    g = scorer_gate()
    r = g["referee"]
    print(f"  control d(acc) = {g['pixels_only_control']['mean_diff']:+.4f}  (must be ~0)")
    print(f"  referee  d(acc) = {r['mean_diff']:+.3f}  CI {r['ci'][0]:+.3f}..{r['ci'][1]:+.3f}")
    print(f"  SCORER GATE: {'PASS' if g['ok'] else 'FAIL'}\n")
    for name, loader in [("xor_tutorial", make_xor), ("breast_cancer", load_breast_cancer)]:
        data = loader(); H, W = grid_dims(data.F)
        rng = np.random.default_rng(0)
        seq = sequential_layout(data.F, H, W)
        base = referee_score(data, seq, H, W)
        print(f"=== {name}  F={data.F} grid={H}x{W}: sequential {base['acc']:.3f}")
        for i in range(3):
            p = referee_paired(data, random_layout(data.F, H, W, rng), seq, H, W)
            tag = "*" if p["ci"][0] > 0 else ("v" if p["ci"][1] < 0 else " ")
            print(f"  random-{i+1}: paired d = {p['mean_diff']:+.3f} "
                  f"CI {p['ci'][0]:+.3f}..{p['ci'][1]:+.3f} {tag}")
        X = np.vstack([data.X_train, data.X_val]); y = np.concatenate([data.y_train, data.y_val])
        ib = interaction_budget(X, y)
        print(f"  dataset gate: gap {ib['gap']:+.3f} -> {ib['verdict']}\n")